In [353]:
import pandas as pd
import numpy as np
import hashlib
from sklearn.neighbors import BallTree
from math import pi

In [308]:
joined_df = pd.read_csv("../processed/joined_transit_data.csv")

/var/folders/vt/__g1vtqn7l13jfkknb257t7r0000gn/T/ipykernel_52636/2390423915.py:1: DtypeWarning: Columns (0,3,5,11,12,16,17,28,31,32,34) have mixed types. Specify dtype option on import or set low_memory=False.
  joined_df = pd.read_csv("../processed/joined_transit_data.csv")


In [309]:
joined_df = joined_df.dropna(subset=['route_id'])

In [310]:
joined_df['route_id'] = joined_df['route_id'].astype(str)
joined_df['trip_id'] = joined_df['trip_id'].astype(str)
joined_df['direction_id'] = joined_df['direction_id'].astype(str)

In [311]:
joined_df["n_stops"] = joined_df.groupby("trip_id")["stop_id"].transform("count")

In [312]:
longest_trips = (
    joined_df[["trip_id", "route_id", "direction_id", "n_stops"]]
    .drop_duplicates()
    .sort_values("n_stops", ascending=False)
    .drop_duplicates(["route_id", "direction_id"])
)

In [313]:
filtered_df = joined_df[
    joined_df["trip_id"].isin(longest_trips["trip_id"])
]

In [314]:
filtered_df = filtered_df[['trip_id', 'stop_id', 'stop_sequence', 'route_id', 'shape_id', 'stop_lat', 'stop_lon', 'arrival_time', 'departure_time', 'direction_id']]
filtered_df = filtered_df.dropna(subset=['route_id', 'stop_sequence'])

In [315]:
sorted_df = filtered_df.sort_values(['route_id', 'direction_id', 'stop_sequence'])

In [316]:
sorted_df["unique_key"] = "stop_" + sorted_df["stop_id"].astype(str) \
    + "_route_" + sorted_df["route_id"].astype(str) \
        + "_direction_" + sorted_df["direction_id"].astype(str)

In [317]:
sorted_df["next_stop"] = sorted_df.groupby(["route_id", "direction_id"])["unique_key"].shift(-1)

In [318]:
def to_seconds(row):
    split = row.split(":")
    return int(split[0]) * 3600 + int(split[1]) * 60 + int(split[2])

sorted_df['arrival_time_seconds'] = sorted_df['arrival_time'].apply(to_seconds)
sorted_df['departure_time_seconds'] = sorted_df['departure_time'].apply(to_seconds)

In [319]:
sorted_df['estimated_travel_time_minutes_between_stops'] = (
    sorted_df.groupby(['route_id', 'direction_id'])['departure_time_seconds'].shift(-1) - sorted_df['arrival_time_seconds']
    ) / 60

In [320]:
# TODO: if folder doesn't exist, create it
sorted_df.to_csv('./transit.csv')

In [321]:
sorted_df[sorted_df['estimated_travel_time_minutes_between_stops'] < 0]

,trip_id,stop_id,stop_sequence,route_id,shape_id,stop_lat,stop_lon,arrival_time,departure_time,direction_id,unique_key,next_stop,arrival_time_seconds,departure_time_seconds,estimated_travel_time_minutes_between_stops


In [338]:
shapes = pd.read_csv('../processed/all_shapes.csv')

/var/folders/vt/__g1vtqn7l13jfkknb257t7r0000gn/T/ipykernel_52636/1589208512.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  shapes = pd.read_csv('../processed/all_shapes.csv')


In [339]:
shapes.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,10002005,47.612137,-122.281769,1,0.0
1,10002005,47.612144,-122.281784,2,5.8
2,10002005,47.612148,-122.281830,3,13.5
3,10002005,47.612141,-122.281853,4,22.0
4,10002005,47.612102,-122.281921,5,45.0


In [340]:
shapes['shape_id'] = shapes['shape_id'].astype(str)
sorted_df['shape_id'] = sorted_df['shape_id'].astype(str)

In [341]:
shapes_merged = sorted_df[['route_id', 'shape_id', 'direction_id']].merge(shapes, how='right', on='shape_id')

In [342]:
shapes_merged = shapes_merged.drop_duplicates()

In [343]:
missing = (
    sorted_df[['route_id','direction_id']]
    .drop_duplicates()
    .merge(
        shapes_merged[['route_id','direction_id']].drop_duplicates(),
        on=['route_id','direction_id'],
        how='left',
        indicator=True
    )
)

missing = missing[missing['_merge'] == 'left_only']
len(missing)


0

In [344]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # meters

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return R * c

In [345]:
shapes = shapes_merged.copy()
shapes["unique_key"] = np.nan

for (route_id, direction_id), stops_group in sorted_df.groupby(["route_id", "direction_id"]):

    # shape points for this route
    shape_points = shapes[
        (shapes["route_id"] == route_id) &
        (shapes["direction_id"] == direction_id)
    ].copy()

    # ensure stop order
    stops_group = stops_group.sort_values("stop_sequence")

    shapes_idx = shape_points.index[0]

    for _, stop in stops_group.iterrows():

        stop_lat = stop["stop_lat"]
        stop_lon = stop["stop_lon"]
        unique_key = stop["unique_key"]

        # only search forward along the shape
        valid_shapes = shape_points.loc[shape_points.index >= shapes_idx]

        if valid_shapes.empty:
            break

        dist = haversine(
            stop_lat,
            stop_lon,
            valid_shapes["shape_pt_lat"],
            valid_shapes["shape_pt_lon"]
        )

        closest_idx = dist.idxmin()

        shapes.loc[closest_idx, "unique_key"] = unique_key

        shapes_idx = closest_idx

/var/folders/vt/__g1vtqn7l13jfkknb257t7r0000gn/T/ipykernel_52636/3424344312.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'stop_LS_N23_T2_route_1-SHUTTLE_direction_0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  shapes.loc[closest_idx, "unique_key"] = unique_key


In [349]:
def lat_lon_to_mercator(lat, lon):
    """Convert lat/lon to Web Mercator projection"""
    k = 6378137
    x = lon * (k * pi/180.0)
    y = np.log(np.tan((90 + lat) * pi/360.0)) * k
    return x, y

In [350]:
shapes['x'], shapes['y'] = lat_lon_to_mercator(
        shapes["shape_pt_lat"].values,
        shapes["shape_pt_lon"].values
    )

In [354]:
def color_from_id(route_id):
    """Generate a color from the route_id"""
    hex_hash = hashlib.md5(str(route_id).encode()).hexdigest()
    return f"#{hex_hash[:6]}"

In [358]:
shapes['color'] = shapes['route_id'].apply(color_from_id)

In [359]:
shapes.head()

,route_id,shape_id,direction_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled,unique_key,x,y,color
0,100089,10002005,1,47.612137,-122.281769,1,0.0,stop_2850_route_100089_direction_1,-1.361234e+07,6.042569e+06,#3f00af
50,100089,10002005,1,47.612144,-122.281784,2,5.8,NaN,-1.361235e+07,6.042570e+06,#3f00af
100,100089,10002005,1,47.612148,-122.281830,3,13.5,NaN,-1.361235e+07,6.042571e+06,#3f00af
150,100089,10002005,1,47.612141,-122.281853,4,22.0,NaN,-1.361235e+07,6.042570e+06,#3f00af
200,100089,10002005,1,47.612102,-122.281921,5,45.0,NaN,-1.361236e+07,6.042563e+06,#3f00af


In [360]:
shapes.to_csv('all_shapes_labeled.csv')

Generate distances between stops

In [21]:
coords = np.radians(sorted_df[['stop_lat', 'stop_lon']])
distance_allowed_miles = .5
tree = BallTree(coords, metric='haversine')
radius = 0.25 / 3959  # 1 mile in radians

indices, dist = tree.query_radius(coords, r=radius, return_distance=True)
dist_miles = dist * 3959

In [22]:
rows = []

node_ids = sorted_df['unique_key'].values
route_ids = sorted_df['route_id'].values

for i, neighbors in enumerate(indices):
    current_id = node_ids[i]
    current_route = route_ids[i]

    for j, neighbor_idx in enumerate(neighbors):
        nearby_id = node_ids[neighbor_idx]
        nearby_route = route_ids[neighbor_idx]

        # skip self
        if current_id == nearby_id:
            continue

        # skip same-route stops
        if pd.notna(current_route) and pd.notna(nearby_route) and current_route == nearby_route:
            continue

        distance = dist_miles[i][j]

        # transfer time: 10 min
        transfer_time = 10
        estimated_time = (distance / 2) * 60 + transfer_time

        rows.append((current_id, nearby_id, estimated_time))

df_nearby_stops = pd.DataFrame(
    rows,
    columns=['source_node', 'nearby_node', 'estimated_time']
)

In [305]:
df_nearby_stops.head()

,source_node,nearby_node,estimated_time
0,stop_LS_N23_T2_route_1-SHUTTLE_direction_0,stop_N23-T1_route_2LINE_direction_1,12.835512
1,stop_LS_N23_T2_route_1-SHUTTLE_direction_0,stop_N23-T2_route_100479_direction_0,11.852196
2,stop_LS_N23_T2_route_1-SHUTTLE_direction_0,stop_N23-T2_route_2LINE_direction_0,11.852196
3,stop_LS_N23_T2_route_1-SHUTTLE_direction_0,stop_N23-T1_route_100479_direction_1,12.835512
4,stop_LS_N19_T2_route_1-SHUTTLE_direction_0,stop_N19-T2_route_2LINE_direction_0,15.993228


In [24]:
df_nearby_stops['pair'] = df_nearby_stops.apply(
    lambda r: tuple(sorted((r['source_node'], r['nearby_node']))),
    axis=1
)

df_nearby_stops = df_nearby_stops.drop_duplicates('pair').drop(columns='pair')

In [306]:
df_nearby_stops.head()

,source_node,nearby_node,estimated_time
0,stop_LS_N23_T2_route_1-SHUTTLE_direction_0,stop_N23-T1_route_2LINE_direction_1,12.835512
1,stop_LS_N23_T2_route_1-SHUTTLE_direction_0,stop_N23-T2_route_100479_direction_0,11.852196
2,stop_LS_N23_T2_route_1-SHUTTLE_direction_0,stop_N23-T2_route_2LINE_direction_0,11.852196
3,stop_LS_N23_T2_route_1-SHUTTLE_direction_0,stop_N23-T1_route_100479_direction_1,12.835512
4,stop_LS_N19_T2_route_1-SHUTTLE_direction_0,stop_N19-T2_route_2LINE_direction_0,15.993228


In [26]:
df_nearby_stops.to_csv("transfers.csv")